## Data Cleaning

- checks for corrupted images 
- resizes and pads to square images 
- saves into a structured cleaned folder 
- use parallel processing with ThreadPoolExecutor 

### Imports 

In [ ]:
from pathlib import Path
from PIL import Image
import cv2
import albumentations as A
from tqdm.auto import tqdm
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
import os 
import sys 
from IPython.display import display # display the image in the notebook

root_path = os.path.abspath(Path(os.getcwd()).parent)
sys.path.append(root_path)

from preprocessing.data_loader import load_deepFashion, load_fashion_ges
from preprocessing.data_cleaning import is_corrupted,resize_and_save, process_image,clean_folder, pad_resize

### Check if an image is corrupted

In [ ]:
# sample image path
path_img = r"D:\03. Projects\technocolabs_projects\1. Deep-Learning-Driven-Virtual-Fashion-Fitting\Data\DeepFashion\Images\img\Zippered_Sweatpants\img_00000001.jpg"
path_img_corrupted = r"D. Projects\technocolabs_projects\1. Deep-Learning-Driven-Virtual-Fashion-Fitting\Data\DeepFashion\Images\img\Zippered_Sweatpants\img_00000001.jpg"

list_paths = [path_img, path_img_corrupted]

for i, path in enumerate(list_paths):
    if is_corrupted(path):
        print(f"image {i}: the image is corrupted")
    else:
        print(f"image {i}: the image isn't corrupted")

### display the image

In [ ]:
with Image.open(path_img) as img:
    display(img)

### resize and pad to (256,256)

In [ ]:
img = cv2.imread(path_img)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

augmented = pad_resize(image=img)
aug_img = augmented['image']

In [ ]:
# before the transformation 
import matplotlib.pyplot as plt

plt.subplot(1,2,1)
plt.imshow(img)
plt.title(f"Original: size {img.shape}")
plt.axis('off')


plt.subplot(1,2,2)
plt.imshow(aug_img)
plt.title(f"Augmented Image: size {aug_img.shape}")
plt.axis('off')

plt.show()

### after augmentation we want to save it into a destination folder 

In [ ]:
# dst_file = Path("Test_save/preprocessed/img_00000001.jpg")

# resize_and_save(path_img, dst_file)

### check if the image is not corrupted resize and save the image into a specific folder and it's ignore it 

In [ ]:
def process_image(row, root_path: Path, data_name: str):
    """
    parameters:
        - row: an iterable form the df annotation that we create from data_loader script
        - root_path: path to source image 
    goals:
        - Check if the image is corrupted:
            - ignore it and return the path
        - else:
            - resize and save that image  into a clean path
            - return the dst path
            
        - remove corrupted rows from the dataframe 
        - replace the old file path with the new cleaned path 
    """
    src = Path(root_path) / row.file
    clean_root = C.CLEAN_DIR / data_name
    
    # if corrupted return its path
    if is_corrupted(src):
        print(f"Corrupted image skipped: {src}")
        return None

    # build the destination folder to copy the image in the dest folder
    dst = clean_root / row.split / row.category.replace(" ", "_") / Path(row.file).name
    resize_and_save(src, dst)
    if data_name == "DeepFashion":
        return {
            "file": str(dst.relative_to(clean_root)),
            "category": row.category,
            "x_1": row.x_1,
            "y_1": row.y_1,
            "x_2": row.x_2,
            "y_2": row.y_2,
            "split": row.split
        }
    elif data_name == "Fashion-gen":
        return {
        "file": str(dst.relative_to(clean_root)),
        "gender": row.gender,
        "masterCategory": row.masterCategory,
        "category": row.category,
        "articleType": row.articleType,
        "baseColour": row.baseColour,
        "season": row.season,
        "year" : row.year,
        "usage": row.usage,
        "productDisplayName": row.productDisplayName,
        "split": row.split
        }

In [19]:
deep_data_path = r"D:\03. Projects\technocolabs_projects\1. Deep-Learning-Driven-Virtual-Fashion-Fitting\Dataset_Clean\DeepFashion\annotation\DeepFashion.csv"
gen_data_path = r"D:\03. Projects\technocolabs_projects\1. Deep-Learning-Driven-Virtual-Fashion-Fitting\Dataset_Clean\FashionGen\annotation\FashionGen.csv"

deepFashion = pd.read_csv(deep_data_path)
deepFashion.head()

,file,category_label,category,category_type,region,x_1,y_1,x_2,y_2,split
0,img/Sheer_Pleated-Front_Blouse/img_00000001.jpg,3,Bomber,1,upper,72,79,232,273,train
1,img/Sheer_Pleated-Front_Blouse/img_00000002.jpg,3,Bomber,1,upper,67,59,155,161,train
2,img/Sheer_Pleated-Front_Blouse/img_00000003.jpg,3,Bomber,1,upper,65,65,156,200,val
3,img/Sheer_Pleated-Front_Blouse/img_00000004.jpg,3,Bomber,1,upper,51,62,167,182,train
4,img/Sheer_Pleated-Front_Blouse/img_00000005.jpg,3,Bomber,1,upper,46,88,166,262,test


In [11]:
len(deepFashion)

289222

In [12]:
Fashion_gen = pd.read_csv(gen_data_path)
Fashion_gen.head()

,file,gender,masterCategory,category,articleType,baseColour,season,year,usage,productDisplayName,split
0,15970.jpg,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011,Casual,Turtle Check Men Navy Blue Shirt,train
1,39386.jpg,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012,Casual,Peter England Men Party Blue Jeans,train
2,59263.jpg,Women,Accessories,Watches,Watches,Silver,Winter,2016,Casual,Titan Women Silver Watch,train
3,21379.jpg,Men,Apparel,Bottomwear,Track Pants,Black,Fall,2011,Casual,Manchester United Men Solid Black Track Pants,train
4,53759.jpg,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012,Casual,Puma Men Grey T-shirt,train


In [15]:
root_path_deep_fashion = r"D:\03. Projects\technocolabs_projects\1. Deep-Learning-Driven-Virtual-Fashion-Fitting\Data\DeepFashion\Images"
root_path_fashion_gen = r"D:\03. Projects\technocolabs_projects\1. Deep-Learning-Driven-Virtual-Fashion-Fitting\Data\Fashion-gen\images"

In [17]:

for i,row in enumerate(deepFashion.itertuples(index=False)):
    if i < 5:
        print(f"row {i + 1}: {row}\n")

row 1: Pandas(file='img/Sheer_Pleated-Front_Blouse/img_00000001.jpg', category_label=3, category='Bomber', category_type=1, region='upper', x_1=72, y_1=79, x_2=232, y_2=273, split='train')

row 2: Pandas(file='img/Sheer_Pleated-Front_Blouse/img_00000002.jpg', category_label=3, category='Bomber', category_type=1, region='upper', x_1=67, y_1=59, x_2=155, y_2=161, split='train')

row 3: Pandas(file='img/Sheer_Pleated-Front_Blouse/img_00000003.jpg', category_label=3, category='Bomber', category_type=1, region='upper', x_1=65, y_1=65, x_2=156, y_2=200, split='val')

row 4: Pandas(file='img/Sheer_Pleated-Front_Blouse/img_00000004.jpg', category_label=3, category='Bomber', category_type=1, region='upper', x_1=51, y_1=62, x_2=167, y_2=182, split='train')

row 5: Pandas(file='img/Sheer_Pleated-Front_Blouse/img_00000005.jpg', category_label=3, category='Bomber', category_type=1, region='upper', x_1=46, y_1=88, x_2=166, y_2=262, split='test')



In [18]:
for i,row in enumerate(tqdm(deepFashion.itertuples(index=False), total=len(deepFashion), desc="preprocessing the images")):
    if i < 5:
        output = process_image(row, root_path_deep_fashion, "DeepFashion")
        

print(output)

preprocessing the images:  81%|████████  | 234314/289222 [00:00<00:00, 1190756.82it/s]

the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly


preprocessing the images: 100%|██████████| 289222/289222 [00:00<00:00, 1186429.48it/s]

{'file': 'test\\Bomber\\img_00000005.jpg', 'category': 'Bomber', 'x_1': 46, 'y_1': 88, 'x_2': 166, 'y_2': 262, 'split': 'test'}


### now we will group all of this together

In [ ]:
# Fashion_gen_cleaned = clean_folder(Fashion_gen, root_path_fashion_gen, "Fashion-gen")
# Fashion_gen_cleaned.head()

the image is saved correctlythe image is saved correctly

the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is s

cleaning/resizing/saving_to_clean_folder:   0%|          | 0/44424 [00:00<?, ?it/s]

the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is s

cleaning/resizing/saving_to_clean_folder:   0%|          | 121/44424 [00:00<00:36, 1204.99it/s]

the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is s

...

the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is s

cleaning/resizing/saving_to_clean_folder:  24%|██▍       | 10678/44424 [00:17<00:42, 801.74it/s]

the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is s

cleaning/resizing/saving_to_clean_folder:  24%|██▍       | 10825/44424 [00:17<00:54, 621.06it/s]


the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is saved correctly
the image is s

In [ ]:
len(Fashion_gen_cleaned)

In [ ]:
deepFashion_cleaned = clean_folder(deepFashion, root_path_deep_fashion, "DeepFashion")
deepFashion_cleaned.head()

In [ ]:
print(len(deepFashion_cleaned))
print(len(Fashion_gen_cleaned))

In [ ]:
from pathlib import Path
IN_ROOT = Path(r"D:\03. Projects\technocolabs_projects\1. Deep-Learning-Driven-Virtual-Fashion-Fitting\Dataset_Clean\DeepFashion")
imgs = list(IN_ROOT.rglob("*.jpg")) + list(IN_ROOT.rglob("*.jpeg")) + list(IN_ROOT.rglob("*.png"))
print(len(imgs))

In [ ]:
IN_ROOT = Path("D:\03. Projects\technocolabs_projects\1. Deep-Learning-Driven-Virtual-Fashion-Fitting\Dataset_Clean\Fashion-gen")
imgs = list(IN_ROOT.rglob("*.jpg")) + list(IN_ROOT.rglob("*.jpeg")) + list(IN_ROOT.rglob("*.png"))
print(len(imgs))